## Итак, Нидлман-Вунш состоит АЖ из 2-х частей
Это прямой и обратный проходы (ПРЯМО как В ГРАДИЕНТОМ СПУСКЕ!)

<img src="https://raw.githubusercontent.com/egichSerg/algorithms-in-bioinfromatics-course/master/img/gradient_meme.jpg" width=300>

In [1]:
# кстати, для работы еще необходимы 2 строки, вот они:
seq_1 = 'ATGCGGGT'
seq_2 = 'TGCCGGT'
# seq_1 = 'TAGCG'
# seq_2 = 'TAAA'

### Начнём с прямого прохода

In [2]:
# 1. Создадим матрицу 
matrix = [[float('-inf') for _ in range(len(seq_1)+1)] for _ in range(len(seq_2)+1)]

# 2. Определим штрафы
match_score = 1
mismatch_penalty = -1
gap_penalty = -1

# 3. Заполним матрицу
# Чтобы не захламлять основной цикл, первую строчку и первый столбец заполним отдельно
matrix[0][0] = ([], 0, None)
for i in range(1, len(seq_2)+1):
    matrix[i][0] = ([(i-1, 0)], matrix[i-1][0][1] + gap_penalty, ["up"])
for j in range(1, len(seq_1)+1):
    matrix[0][j] = ([(0, j-1)], matrix[0][j-1][1] + gap_penalty, ["left"])

# теперь можно)
for i in range(1, len(seq_2)+1):
    for j in range(1, len(seq_1)+1):
        diag_score = matrix[i-1][j-1][1] + (match_score if seq_1[j-1] == seq_2[i-1] else mismatch_penalty)
        max_score = max(matrix[i-1][j][1] + gap_penalty,
                        matrix[i][j-1][1] + gap_penalty,
                        diag_score)
        ancestors = [] # запоминаем откуда пришли
        read_ancestors = []
        if matrix[i-1][j][1] + gap_penalty == max_score:
            ancestors.append((i-1, j))
            read_ancestors.append("up")
        if matrix[i][j-1][1] + gap_penalty == max_score:
            ancestors.append((i, j-1))
            read_ancestors.append("left")
        if diag_score == max_score:
            ancestors.append((i-1, j-1))
            read_ancestors.append("diag")

        matrix[i][j] = (ancestors, max_score, read_ancestors)

### Выведем таблицу в приятно-читаемом виде

In [3]:
import pandas as pd
read_matrix = pd.DataFrame(matrix).map(lambda x: (x[2], x[1]))
read_matrix = read_matrix.rename(
    index={i: seq_2[i-1] for i in range(1, len(seq_2)+1)},
    columns={j: seq_1[j-1] for j in range(1, len(seq_1)+1)})

### И наблюдаем следующую организацию: для каждой ячейки есть значение максимального скора и откуда пришли

In [4]:
# pd.DataFrame(matrix)
read_matrix

,0,A,T,G,C,G,G,G,T
0,"(None, 0)","([left], -1)","([left], -2)","([left], -3)","([left], -4)","([left], -5)","([left], -6)","([left], -7)","([left], -8)"
T,"([up], -1)","([diag], -1)","([diag], 0)","([left], -1)","([left], -2)","([left], -3)","([left], -4)","([left], -5)","([left, diag], -6)"
G,"([up], -2)","([up, diag], -2)","([up], -1)","([diag], 1)","([left], 0)","([left, diag], -1)","([left, diag], -2)","([left, diag], -3)","([left], -4)"
C,"([up], -3)","([up, diag], -3)","([up], -2)","([up], 0)","([diag], 2)","([left], 1)","([left], 0)","([left], -1)","([left], -2)"
C,"([up], -4)","([up, diag], -4)","([up], -3)","([up], -1)","([up, diag], 1)","([diag], 1)","([left, diag], 0)","([left, diag], -1)","([left, diag], -2)"
G,"([up], -5)","([up, diag], -5)","([up], -4)","([up, diag], -2)","([up], 0)","([diag], 2)","([diag], 2)","([left, diag], 1)","([left], 0)"
G,"([up], -6)","([up, diag], -6)","([up], -5)","([up, diag], -3)","([up], -1)","([up, diag], 1)","([diag], 3)","([diag], 3)","([left], 2)"
T,"([up], -7)","([up, diag], -7)","([diag], -5)","([up], -4)","([up], -2)","([up], 0)","([up], 2)","([up, diag], 2)","([diag], 4)"


## Похоже, пришло время восстановить пути (выравнивания)
Иначе зачем мы запоминали их?

In [6]:
# Начиная с конца пытаемся пройти по всем возможным путям
# тут можно не засирать стэк, но мне строго лень делать НЕ рекурсивно. Тем более рекурсивно более читаемо
def get_path(i, j, seq1, seq2):
    if matrix[i][j][2] is None:  # условие остановки
        print("="*20, "\n", seq1[::-1], "\n", seq2[::-1])
        return None
    for ancestor, ancestor_type in zip(matrix[i][j][0], matrix[i][j][2]):
        if ancestor_type == "up":
            seq1 += "-"
            seq2 += seq_2[i-1]
        elif ancestor_type == "left":
            seq1 += seq_1[j-1]
            seq2 += "-"
        elif ancestor_type == "diag":
            seq1 += seq_1[j-1]
            seq2 += seq_2[i-1]
        get_path(*ancestor, seq1, seq2)


get_path(len(seq_2), len(seq_1), "", "")
print(f"\n\nСкор для всех выравниваний составил {matrix[len(seq_2)][len(seq_1)][1]}")

 ATGCGGGT 
 -TGCCGGT


Скор для всех выравниваний составил 4
